## Spatial Signal Strength Analysis — Chandler Fashion Center
Analyze cell signal quality for points that fall inside the Chandler Fashion Center polygon using Databricks Spatial SQL (`ST_*` functions).

In [0]:
# ========================== CONFIGURATION ==========================
# Update these values for your environment. Everything else in this
# notebook derives from them.
# ===================================================================

# Unity Catalog location
CATALOG = "cmegdemos_catalog"
SCHEMA  = "geospatial_analytics"

# Volume path containing the raw CSV and shapefile
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"

# File names (assumed identical across environments)
CSV_FILE = "Echolocate points around Chandler Fashion Center Area (0.5km).csv"
SHP_FILE = "Chandler Fashion Center Area (0.5km).shp"

# Target Delta table for persisted intersected points
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.intersected_signal_points"

# SQL warehouse used by the Dash app
WAREHOUSE_ID = "9cd919d96b11bf1c"

# Databricks App name (for programmatic deploy)
APP_NAME = "geospatial-heat-map-viz-test"

# App source directory (relative to this notebook's folder)
import os
_nb_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
APP_DIR = f"/Workspace{_nb_dir}/signal_quality_app"

print(f"Catalog/Schema : {CATALOG}.{SCHEMA}")
print(f"Volume path    : {VOLUME_PATH}")
print(f"Target table   : {TARGET_TABLE}")
print(f"Warehouse ID   : {WAREHOUSE_ID}")
print(f"App directory  : {APP_DIR}")

In [0]:
# Read signal-point CSV — use escape='"' for RFC-4180 double-quote escaping
csv_path = f"{VOLUME_PATH}/{CSV_FILE}"

points_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("escape", '"')          # handles "" inside quoted fields
    .load(csv_path)
)

points_df.createOrReplaceTempView("raw_points")
print(f"Point records: {points_df.count():,}")
points_df.printSchema()

In [0]:
%pip install geopandas --quiet

In [0]:
import geopandas as gpd

shp_path = f"{VOLUME_PATH}/{SHP_FILE}"
gdf = gpd.read_file(shp_path)
print(f"Polygon records: {len(gdf)}")
print(f"CRS: {gdf.crs}")
print(gdf.head())

# Convert geometry to WKT so Spark SQL can ingest it
gdf["wkt"] = gdf.geometry.apply(lambda g: g.wkt)
cols_to_keep = [c for c in gdf.columns if c != "geometry"]

polygons_sdf = spark.createDataFrame(gdf[cols_to_keep])
polygons_sdf.createOrReplaceTempView("raw_polygons")
display(polygons_sdf)

In [0]:
%sql
-- Spatial join: find signal points that fall inside building polygons
-- Uses ST_Contains with GEOMETRY(4326) for both sides
CREATE OR REPLACE TEMP VIEW intersected_points AS
SELECT /*+ BROADCAST(poly) */
    p.LONGITUDE,
    p.LATITUDE,
    p.EVENTDATE,
    p.ED_TIMESTAMP,
    p.ED_ENVIRONMENT_TELEPHONY_PRIMARYCELL_CELLSIGNALSTRENGTH_RSRP   AS rsrp,
    p.ED_ENVIRONMENT_NET_CONNECTEDWIFISTATUS_RSSILEVEL               AS wifi_rssi,
    p.ED_ENVIRONMENT_TELEPHONY_PRIMARYCELL_CELLIDENTITY_NETWORKNAME  AS network_name,
    p.ED_ENVIRONMENT_TELEPHONY_NETWORKTYPE                           AS network_type,
    p.ED_ENVIRONMENT_NET_CONNECTIVITYTYPE                            AS connectivity_type,
    p.ED_ENVIRONMENT_TELEPHONY_PRIMARYCELL_CELLTYPE                  AS cell_type,
    p.ED_ENVIRONMENT_TELEPHONY_SERVICESTATE                         AS service_state,
    poly.osm_id        AS building_osm_id,
    poly.fclass         AS building_class,
    poly.name           AS building_name,
    poly.type           AS building_type
FROM raw_points p
JOIN raw_polygons poly
  ON ST_Contains(
       ST_GeomFromWKT(poly.wkt, 4326),
       ST_Point(p.LONGITUDE, p.LATITUDE, 4326)
     );

SELECT count(*) AS intersected_count FROM intersected_points;

In [0]:
%sql
-- Preview the intersected points
SELECT * FROM intersected_points LIMIT 20;

In [0]:
%sql
-- Signal strength analysis by building
SELECT
    building_osm_id,
    building_name,
    building_type,
    COUNT(*)                        AS point_count,
    ROUND(AVG(rsrp), 1)            AS avg_rsrp,
    MIN(rsrp)                       AS min_rsrp,
    MAX(rsrp)                       AS max_rsrp,
    ROUND(AVG(wifi_rssi), 1)       AS avg_wifi_rssi,
    -- Classify signal quality based on RSRP (3GPP thresholds)
    ROUND(100.0 * SUM(CASE WHEN rsrp >= -80  THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_excellent,
    ROUND(100.0 * SUM(CASE WHEN rsrp BETWEEN -90 AND -81 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_good,
    ROUND(100.0 * SUM(CASE WHEN rsrp BETWEEN -100 AND -91 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_fair,
    ROUND(100.0 * SUM(CASE WHEN rsrp < -100 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_poor
FROM intersected_points
WHERE rsrp IS NOT NULL
GROUP BY building_osm_id, building_name, building_type
ORDER BY point_count DESC;

In [0]:
%sql
-- Signal quality by network type and connectivity
SELECT
    network_type,
    connectivity_type,
    cell_type,
    COUNT(*)                  AS readings,
    ROUND(AVG(rsrp), 1)      AS avg_rsrp,
    ROUND(AVG(wifi_rssi), 1) AS avg_wifi_rssi
FROM intersected_points
WHERE rsrp IS NOT NULL
GROUP BY network_type, connectivity_type, cell_type
ORDER BY readings DESC;

In [0]:
%sql
-- Signal strength over time (daily trend)
SELECT
    EVENTDATE,
    COUNT(*)             AS readings,
    ROUND(AVG(rsrp), 1) AS avg_rsrp,
    MIN(rsrp)            AS min_rsrp,
    MAX(rsrp)            AS max_rsrp
FROM intersected_points
WHERE rsrp IS NOT NULL AND EVENTDATE IS NOT NULL
GROUP BY EVENTDATE
ORDER BY EVENTDATE;

Databricks visualization. Run in Databricks to view.

In [0]:
%pip install folium --quiet

In [0]:
# Heatmap of call quality (RSRP signal strength) for intersected points
import folium
from folium.plugins import HeatMap
import pandas as pd

# Fetch intersected points with signal strength
heat_df = spark.sql("""
    SELECT LATITUDE, LONGITUDE, rsrp, wifi_rssi,
           building_osm_id, network_type, connectivity_type
    FROM intersected_points
    WHERE rsrp IS NOT NULL
""").toPandas()

print(f"Heatmap data points: {len(heat_df):,}")

# Center map on the data centroid
center_lat = heat_df["LATITUDE"].mean()
center_lon = heat_df["LONGITUDE"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles="CartoDB dark_matter")

# Normalize RSRP to 0-1 for heatmap intensity
# RSRP typically ranges from -140 (worst) to -44 (best)
# Invert so stronger signal = higher weight
heat_df["weight"] = (heat_df["rsrp"] + 140) / 96  # maps -140→0, -44→1
heat_df["weight"] = heat_df["weight"].clip(0, 1)

heat_data = heat_df[["LATITUDE", "LONGITUDE", "weight"]].values.tolist()

HeatMap(
    heat_data,
    radius=14,
    blur=10,
    max_zoom=18,
    gradient={0.2: "blue", 0.4: "cyan", 0.6: "lime", 0.8: "yellow", 1.0: "red"},
).add_to(m)

# Add building polygon outlines
import json
for _, row in gdf.iterrows():
    geo_json = json.loads(gpd.GeoSeries([row.geometry]).to_json())
    folium.GeoJson(
        geo_json,
        style_function=lambda x: {
            "fillColor": "transparent",
            "color": "white",
            "weight": 1,
            "fillOpacity": 0,
        },
    ).add_to(m)

m

In [0]:
# Map individual intersected points color-coded by signal quality
import matplotlib.colors as mcolors

m2 = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles="CartoDB positron")

# Classify RSRP into quality buckets
def rsrp_color(rsrp):
    if rsrp >= -80:
        return "green"       # Excellent
    elif rsrp >= -90:
        return "blue"        # Good
    elif rsrp >= -100:
        return "orange"      # Fair
    else:
        return "red"         # Poor

# Sample for performance (plot up to 5000 points)
sample = heat_df.sample(n=min(5000, len(heat_df)), random_state=42)

for _, row in sample.iterrows():
    folium.CircleMarker(
        location=[row["LATITUDE"], row["LONGITUDE"]],
        radius=3,
        color=rsrp_color(row["rsrp"]),
        fill=True,
        fill_opacity=0.7,
        popup=f"RSRP: {row['rsrp']} dBm<br>Network: {row['network_type']}<br>Building: {row['building_osm_id']}",
    ).add_to(m2)

# Building outlines
for _, row in gdf.iterrows():
    geo_json = json.loads(gpd.GeoSeries([row.geometry]).to_json())
    folium.GeoJson(
        geo_json,
        style_function=lambda x: {
            "fillColor": "#333333",
            "color": "black",
            "weight": 1.5,
            "fillOpacity": 0.15,
        },
    ).add_to(m2)

# Legend
legend_html = """
<div style="position:fixed; bottom:50px; left:50px; z-index:1000;
     background:white; padding:10px; border-radius:5px; border:1px solid grey;">
<b>Signal Quality (RSRP)</b><br>
<i style="background:green;width:12px;height:12px;display:inline-block;"></i> Excellent (≥ -80 dBm)<br>
<i style="background:blue;width:12px;height:12px;display:inline-block;"></i> Good (-90 to -80)<br>
<i style="background:orange;width:12px;height:12px;display:inline-block;"></i> Fair (-100 to -90)<br>
<i style="background:red;width:12px;height:12px;display:inline-block;"></i> Poor (< -100 dBm)
</div>
"""
m2.get_root().html.add_child(folium.Element(legend_html))

m2

In [0]:
# Persist intersected points to Delta for the Dash app to query
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

spark.sql(f"""
    CREATE TABLE {TARGET_TABLE} AS
    SELECT * FROM intersected_points
""")

print(f"Saved {spark.table(TARGET_TABLE).count():,} rows to {TARGET_TABLE}")

## Dash App — Signal Quality Heatmap

The interactive app is generated into the `signal_quality_app/` subdirectory by the cell below.
All environment-specific values (catalog, schema, warehouse ID) are injected from the
**Configuration** cell at the top of this notebook — edit that cell for your environment.

**App files:**
| File | Purpose |
| --- | --- |
| `app.py` | Dash app with heatmap, point map, filters, and stats |
| `requirements.txt` | Python dependencies (dash, plotly, databricks-sdk, databricks-sql-connector) |
| `app.yaml` | Databricks App deployment config |

**Auth:** The app uses `WorkspaceClient()` from the Databricks SDK, which auto-detects
credentials via the app’s managed service principal — no hostname or token env vars needed.

**Interactive controls:**
* **View Mode**: toggle between density heatmap and color-coded point map
* **Heatmap Metric**: aggregate by RSRP signal strength or WiFi RSSI
* **Color By** (point map): Signal Quality, Network Type, Connectivity, Cell Type, or Building
* **Filters**: Network Type, Connectivity Type, Cell Type, Building, Signal Quality, Date Range

**Prerequisites / Deployment (all done via code below):**
1. **Grant the app service principal** SELECT on the Delta table and CAN USE on the SQL warehouse.
2. Run the *Generate app files* cell below — it writes `app.py`, `requirements.txt`, and `app.yaml` using the parameters defined in the Configuration cell.
3. Deploy via *Compute → Apps → Create App* → point to `signal_quality_app/`, or run the deploy cell at the bottom.

In [0]:
# Generate the Dash app files, injecting TARGET_TABLE and WAREHOUSE_ID
# from the Configuration cell so the app is ready for any environment.

os.makedirs(APP_DIR, exist_ok=True)

# ── app.py ───────────────────────────────────────────────────────────────────
app_py = f'''
import os
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks import sql

# Configuration (injected from notebook parameters)
TABLE = "{TARGET_TABLE}"
WAREHOUSE_ID = os.getenv("DATABRICKS_WAREHOUSE_ID", "{WAREHOUSE_ID}")
MAP_CENTER = {{"lat": 33.3013, "lon": -111.8986}}
DEFAULT_ZOOM = 15

w = WorkspaceClient()


def get_connection():
    server_hostname = w.config.host.replace("https://", "").rstrip("/")
    http_path = f"/sql/1.0/warehouses/{{WAREHOUSE_ID}}"
    return sql.connect(
        server_hostname=server_hostname,
        http_path=http_path,
        credentials_provider=lambda: w.config.authenticate,
    )


def load_data() -> pd.DataFrame:
    query = f"""SELECT LONGITUDE, LATITUDE, EVENTDATE, rsrp, wifi_rssi,
               network_name, network_type, connectivity_type,
               cell_type, service_state,
               building_osm_id, building_name, building_type
        FROM {{TABLE}} WHERE rsrp IS NOT NULL"""
    with get_connection() as conn:
        df = pd.read_sql(query, conn)
    # Clean up verbose prefixes for display
    df["network_type"] = df["network_type"].str.replace("NETWORK_TYPE_", "", regex=False)
    df["signal_quality"] = pd.cut(
        df["rsrp"],
        bins=[float("-inf"), -100, -90, -80, float("inf")],
        labels=["Poor", "Fair", "Good", "Excellent"], ordered=True,
    )
    df["building_label"] = df["building_name"].fillna("ID:" + df["building_osm_id"].astype(str))
    return df


print("Loading data from Delta table...")
df = load_data()
print(f"Loaded {{len(df):,}} signal readings.")


def opts(series):
    vals = sorted(series.dropna().unique())
    return [{{"label": v, "value": v}} for v in vals]


METRIC_OPTIONS = [{{"label": "RSRP Signal Strength", "value": "rsrp"}},
                  {{"label": "WiFi RSSI Level", "value": "wifi_rssi"}}]
VIEW_OPTIONS = [{{"label": "Heatmap", "value": "heatmap"}},
                {{"label": "Point Map", "value": "scatter"}}]
COLOR_BY_OPTIONS = [
    {{"label": "Signal Quality", "value": "signal_quality"}},
    {{"label": "Network Type", "value": "network_type"}},
    {{"label": "Connectivity Type", "value": "connectivity_type"}},
    {{"label": "Cell Type", "value": "cell_type"}},
    {{"label": "Building", "value": "building_label"}},
]
QUALITY_COLORS = {{"Excellent": "#2ecc71", "Good": "#3498db",
                   "Fair": "#f39c12", "Poor": "#e74c3c"}}

app = dash.Dash(__name__, title="Signal Quality")

app.index_string = \'\'\'\n<!DOCTYPE html>
<html>
    <head>
        {{%metas%}}
        <title>{{%title%}}</title>
        {{%favicon%}}
        {{%css%}}
        <style>
            .Select-menu-outer, .Select-menu-outer * {{ color: #111 !important; }}
            .VirtualizedSelectOption {{ color: #111 !important; background-color: #fff !important; }}
            .VirtualizedSelectFocusedOption {{ color: #111 !important; background-color: #deebff !important; }}
            .Select-option {{ color: #111 !important; }}
            .Select-option.is-focused {{ color: #111 !important; background-color: #deebff !important; }}
            .Select-value-label, .Select--multi .Select-value-label {{ color: #111 !important; }}
            .Select-value {{ color: #111 !important; }}
            .Select--single > .Select-control .Select-value {{ color: #111 !important; }}
            .Select-placeholder, .Select-input > input::placeholder {{ color: #999 !important; }}
            .Select-input > input {{ color: #eee !important; }}
            .Select--multi .Select-value {{ background-color: #e8e8e8 !important; border-color: #ccc !important; color: #111 !important; }}
            .Select-noresults {{ color: #666 !important; }}
            .DateInput_input {{ color: #111 !important; }}
        </style>
    </head>
    <body>
        {{%app_entry%}}
        <footer>{{%config%}}{{%scripts%}}{{%renderer%}}</footer>
    </body>
</html>
\'\'\'\n
SIDEBAR = {{"width": "310px", "minWidth": "310px", "padding": "20px",
            "backgroundColor": "#1a1a2e", "color": "#eee",
            "borderRadius": "8px", "overflowY": "auto", "maxHeight": "85vh"}}
LBL = {{"fontWeight": "600", "marginTop": "14px", "marginBottom": "4px", "fontSize": "13px"}}
DD = {{"marginBottom": "8px"}}

app.layout = html.Div([
    html.Div([
        html.H2("Cell Signal Quality \u2014 Chandler Fashion Center",
                style={{"margin": "0", "color": "#1a1a2e"}}),
        html.P("Interactive spatial heatmap of RSRP signal strength inside building polygons",
               style={{"margin": "4px 0 0", "color": "#666", "fontSize": "14px"}}),
    ], style={{"padding": "16px 24px", "borderBottom": "2px solid #e0e0e0"}}),
    html.Div([
        html.Div([
            html.H4("Controls", style={{"color": "#e94560", "marginTop": "0"}}),
            html.Label("View Mode", style=LBL),
            dcc.RadioItems(id="view-mode", options=VIEW_OPTIONS, value="heatmap",
                           labelStyle={{"display": "block", "marginBottom": "4px"}},
                           style={{"marginBottom": "8px"}}),
            html.Label("Heatmap Metric", style=LBL),
            dcc.Dropdown(id="metric", options=METRIC_OPTIONS, value="rsrp",
                         clearable=False, style=DD),
            html.Div(id="color-by-container", children=[
                html.Label("Color By (Point Map)", style=LBL),
                dcc.Dropdown(id="color-by", options=COLOR_BY_OPTIONS,
                             value="signal_quality", clearable=False, style=DD),
            ], style={{"display": "none"}}),
            html.Hr(style={{"borderColor": "#333"}}),
            html.H4("Filters", style={{"color": "#e94560"}}),
            html.Label("Network Type", style=LBL),
            dcc.Dropdown(id="f-network", options=opts(df["network_type"]),
                         multi=True, placeholder="All", style=DD),
            html.Label("Connectivity", style=LBL),
            dcc.Dropdown(id="f-connectivity", options=opts(df["connectivity_type"]),
                         multi=True, placeholder="All", style=DD),
            html.Label("Cell Type", style=LBL),
            dcc.Dropdown(id="f-cell", options=opts(df["cell_type"]),
                         multi=True, placeholder="All", style=DD),
            html.Label("Building", style=LBL),
            dcc.Dropdown(id="f-building", options=opts(df["building_label"]),
                         multi=True, placeholder="All", style=DD),
            html.Label("Signal Quality", style=LBL),
            dcc.Dropdown(id="f-quality",
                         options=[{{"label": q, "value": q}}
                                  for q in ["Excellent", "Good", "Fair", "Poor"]],
                         multi=True, placeholder="All", style=DD),
            html.Label("Date Range", style=LBL),
            dcc.DatePickerRange(id="f-dates",
                                min_date_allowed=df["EVENTDATE"].min(),
                                max_date_allowed=df["EVENTDATE"].max(),
                                start_date=df["EVENTDATE"].min(),
                                end_date=df["EVENTDATE"].max(),
                                style={{"marginBottom": "14px"}}),
            html.Hr(style={{"borderColor": "#333"}}),
            html.Div(id="stats-panel"),
        ], style=SIDEBAR),
        html.Div([dcc.Graph(id="map-graph", style={{"height": "82vh"}})],
                 style={{"flex": "1", "paddingLeft": "16px"}}),
    ], style={{"display": "flex", "padding": "16px", "gap": "0"}}),
], style={{"fontFamily": "'Segoe UI', Arial, sans-serif",
          "backgroundColor": "#f5f5f5", "minHeight": "100vh"}})


@app.callback(Output("color-by-container", "style"), Input("view-mode", "value"))
def toggle_color_by(view):
    return {{"display": "block"}} if view == "scatter" else {{"display": "none"}}


@app.callback(
    [Output("map-graph", "figure"), Output("stats-panel", "children")],
    [Input("view-mode", "value"), Input("metric", "value"),
     Input("color-by", "value"), Input("f-network", "value"),
     Input("f-connectivity", "value"), Input("f-cell", "value"),
     Input("f-building", "value"), Input("f-quality", "value"),
     Input("f-dates", "start_date"), Input("f-dates", "end_date")],
)
def update_map(view, metric, color_by, networks, connectivities, cells,
               buildings, qualities, start_date, end_date):
    filtered = df.copy()
    if networks:       filtered = filtered[filtered["network_type"].isin(networks)]
    if connectivities: filtered = filtered[filtered["connectivity_type"].isin(connectivities)]
    if cells:          filtered = filtered[filtered["cell_type"].isin(cells)]
    if buildings:      filtered = filtered[filtered["building_label"].isin(buildings)]
    if qualities:      filtered = filtered[filtered["signal_quality"].isin(qualities)]
    if start_date:     filtered = filtered[filtered["EVENTDATE"] >= pd.to_datetime(start_date).date()]
    if end_date:       filtered = filtered[filtered["EVENTDATE"] <= pd.to_datetime(end_date).date()]

    center = ({{"lat": filtered["LATITUDE"].mean(), "lon": filtered["LONGITUDE"].mean()}}
              if len(filtered) > 0 else MAP_CENTER)

    if len(filtered) == 0:
        fig = px.scatter_mapbox(
            pd.DataFrame({{"lat": [MAP_CENTER["lat"]], "lon": [MAP_CENTER["lon"]]}}),
            lat="lat", lon="lon", zoom=DEFAULT_ZOOM, mapbox_style="carto-darkmatter")
        fig.update_layout(title="No data matches the current filters")
    elif view == "heatmap":
        plot_df = filtered.dropna(subset=[metric]).copy()
        if metric == "rsrp":  plot_df["_z"] = (plot_df["rsrp"] + 140) / 96
        else:                 plot_df["_z"] = (plot_df["wifi_rssi"] + 100) / 60
        plot_df["_z"] = plot_df["_z"].clip(0, 1)
        fig = px.density_mapbox(
            plot_df, lat="LATITUDE", lon="LONGITUDE", z="_z",
            radius=14, center=center, zoom=DEFAULT_ZOOM,
            mapbox_style="carto-darkmatter",
            color_continuous_scale=["#0d0887", "#46039f", "#7201a8", "#9c179e",
                                   "#bd3786", "#d8576b", "#ed7953", "#fb9f3a",
                                   "#fdca26", "#f0f921"])
        fig.update_layout(coloraxis_colorbar_title="RSRP (dBm)" if metric == "rsrp" else "WiFi RSSI (dBm)")
    else:
        sample = filtered.sample(n=min(8000, len(filtered)), random_state=42)
        fig = px.scatter_mapbox(
            sample, lat="LATITUDE", lon="LONGITUDE", color=color_by,
            hover_data=["rsrp", "wifi_rssi", "network_type", "building_label"],
            center=center, zoom=DEFAULT_ZOOM, mapbox_style="carto-darkmatter",
            color_discrete_map=(QUALITY_COLORS if color_by == "signal_quality" else None),
            category_orders={{"signal_quality": ["Excellent", "Good", "Fair", "Poor"]}},
            opacity=0.7)
    fig.update_layout(margin=dict(l=0, r=0, t=30, b=0))

    n = len(filtered)
    if n > 0:
        avg_rsrp = filtered["rsrp"].mean()
        q_counts = filtered["signal_quality"].value_counts()
        stats = [
            html.H4("Summary", style={{"color": "#e94560", "marginTop": "0"}}),
            html.P(f"Points: {{n:,}}", style={{"margin": "4px 0"}}),
            html.P(f"Avg RSRP: {{avg_rsrp:.1f}} dBm", style={{"margin": "4px 0"}}),
            html.Hr(style={{"borderColor": "#333"}}),
            html.P(f"Excellent: {{q_counts.get(\'Excellent\', 0):,}} ({{q_counts.get(\'Excellent\', 0)/n*100:.1f}}%)",
                   style={{"margin": "2px 0", "color": "#2ecc71"}}),
            html.P(f"Good: {{q_counts.get(\'Good\', 0):,}} ({{q_counts.get(\'Good\', 0)/n*100:.1f}}%)",
                   style={{"margin": "2px 0", "color": "#3498db"}}),
            html.P(f"Fair: {{q_counts.get(\'Fair\', 0):,}} ({{q_counts.get(\'Fair\', 0)/n*100:.1f}}%)",
                   style={{"margin": "2px 0", "color": "#f39c12"}}),
            html.P(f"Poor: {{q_counts.get(\'Poor\', 0):,}} ({{q_counts.get(\'Poor\', 0)/n*100:.1f}}%)",
                   style={{"margin": "2px 0", "color": "#e74c3c"}}),
        ]
    else:
        stats = [html.P("No data", style={{"color": "#888"}})]
    return fig, html.Div(stats)


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8050, debug=False)
'''

with open(os.path.join(APP_DIR, "app.py"), "w") as f:
    f.write(app_py.lstrip())
print(f"Wrote app.py  ({len(app_py):,} chars)")

# ── requirements.txt ─────────────────────────────────────────────────────────
reqs = """dash>=2.14.0
plotly>=5.18.0
pandas>=2.0.0
databricks-sql-connector>=3.0.0
databricks-sdk>=0.20.0
"""
with open(os.path.join(APP_DIR, "requirements.txt"), "w") as f:
    f.write(reqs)
print("Wrote requirements.txt")

# ── app.yaml ─────────────────────────────────────────────────────────────────
app_yaml = f"""command:
  - "python"
  - "app.py"

env:
  - name: "DATABRICKS_WAREHOUSE_ID"
    description: "SQL warehouse ID used to query the Delta table"
    value: "{WAREHOUSE_ID}"
"""
with open(os.path.join(APP_DIR, "app.yaml"), "w") as f:
    f.write(app_yaml)
print("Wrote app.yaml")

print(f"\nAll app files generated in {APP_DIR}")

In [0]:
# Create the Databricks App and wait for compute to be ready
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import App
import time

w = WorkspaceClient()

# Check if app already exists
try:
    existing = w.apps.get(APP_NAME)
    print(f"App '{APP_NAME}' already exists (status: {existing.compute_status})")
    app_info = existing
except Exception:
    print(f"Creating app '{APP_NAME}'...")
    app_info = w.apps.create_and_wait(
        app=App(name=APP_NAME, description="Interactive signal quality heatmap for Chandler Fashion Center"),
    )
    print(f"App created successfully.")

print(f"App URL: {app_info.url}")

# Retrieve the app's service principal ID for the next step
sp_id = app_info.service_principal_id
sp_name = app_info.service_principal_name
print(f"Service principal: {sp_name} (ID: {sp_id})")

In [0]:
# Grant the app service principal access to warehouse and table
import time

# Look up the SP's application_id (UUID) — required for both UC and permissions API
sp_details = w.service_principals.get(sp_id)
sp_app_id = sp_details.application_id
print(f"SP display name  : {sp_name}")
print(f"SP application_id: {sp_app_id}")

# Small delay to let the SP propagate across services
time.sleep(5)

# 1) CAN_USE on the SQL warehouse — use application_id (UUID), not display name
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel
w.permissions.update(
    request_object_type="sql/warehouses",
    request_object_id=WAREHOUSE_ID,
    access_control_list=[
        AccessControlRequest(
            service_principal_name=sp_app_id,
            permission_level=PermissionLevel.CAN_USE,
        )
    ],
)
print(f"Granted CAN_USE on warehouse {WAREHOUSE_ID} to SP '{sp_app_id}'")

# 2) SELECT on the Delta table via Unity Catalog
spark.sql(f"GRANT SELECT ON TABLE {TARGET_TABLE} TO `{sp_app_id}`")
print(f"Granted SELECT on {TARGET_TABLE} to SP '{sp_app_id}'")

# 3) USAGE on catalog and schema so SP can resolve the table path
try:
    spark.sql(f"GRANT USE CATALOG ON CATALOG `{CATALOG}` TO `{sp_app_id}`")
    print(f"Granted USE CATALOG on {CATALOG}")
except Exception as e:
    print(f"Catalog grant note: {e}")
try:
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA `{CATALOG}`.`{SCHEMA}` TO `{sp_app_id}`")
    print(f"Granted USE SCHEMA on {CATALOG}.{SCHEMA}")
except Exception as e:
    print(f"Schema grant note: {e}")

In [0]:
# Deploy the app with the generated source code
from databricks.sdk.service.apps import AppDeployment

print(f"Deploying '{APP_NAME}' from {APP_DIR}...")
result = w.apps.deploy(
    app_name=APP_NAME,
    app_deployment=AppDeployment(source_code_path=APP_DIR),
).result()

print(f"Deployment ID : {result.deployment_id}")
print(f"Status        : {result.status}")
print(f"\nApp live at   : {app_info.url}")